# 04 — Baseline Model

**Goal:** Establish a performance floor that every model in notebook 05 must beat.

> ✅ Only **train** data. Test set is still locked.

**Two baselines:**
1. **Naive baseline** — predict last week same hour (`lag_168h`). No ML.
2. **Ridge baseline** — linear model with calendar + weather features only (no lags).
3. **Ridge + lags** — linear model with full feature set. Real baseline.

All logged to MLflow. We use `TimeSeriesSplit` — never `KFold`.

In [1]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import json
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.base import BaseEstimator, TransformerMixin

PROCESSED_PATH    = '../data/processed/'
MLFLOW_EXPERIMENT = 'energy-demand-prediction'
TARGET            = 'total load actual'

# TimeSeriesSplit config
# gap=24: skip 24h between train and val in each fold
# This prevents adjacent-hour leakage in cross-validation
TSCV = TimeSeriesSplit(n_splits=5, gap=24)

d:\my_projects\Energy_demand_predictor\energy_demand\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load Data & Feature Config

In [2]:
train_raw = pd.read_csv(PROCESSED_PATH + 'train.csv')
train_raw['time'] = pd.to_datetime(train_raw['time'], utc=True)

with open(PROCESSED_PATH + 'feature_config.json') as f:
    config = json.load(f)

ALL_FEATURES = config['all_features']
LAG_FEATURES = config['lag_features']

print(f'Train shape: {train_raw.shape}')
print(f'Total features: {len(ALL_FEATURES)}')

Train shape: (24544, 32)
Total features: 40


## 2. Define the Feature Transformer (from notebook 03)

In [3]:
class EnergyFeatureTransformer(BaseEstimator, TransformerMixin):
    """Full feature engineering — copy from notebook 03."""

    def __init__(self, lag_hours=None, add_cyclical=True):
        self.lag_hours    = lag_hours or [1, 24, 48, 168, 336]
        self.add_cyclical = add_cyclical

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.copy()
        df['time'] = pd.to_datetime(df['time'], utc=True)
        df = df.sort_values('time').reset_index(drop=True)
        df = self._add_calendar(df)
        if self.add_cyclical:
            df = self._add_cyclical(df)
        df = self._add_lags(df)
        df = self._add_rolling(df)
        df = self._add_weather(df)
        df = df.dropna().reset_index(drop=True)
        return df

    def _add_calendar(self, df):
        t = df['time']
        df['hour']        = t.dt.hour
        df['dow']         = t.dt.dayofweek
        df['month']       = t.dt.month
        df['day_of_year'] = t.dt.dayofyear
        df['week_of_year']= t.dt.isocalendar().week.astype(int)
        df['quarter']     = t.dt.quarter
        df['is_weekend']  = (t.dt.dayofweek >= 5).astype(int)
        season_map        = {12:3,1:3,2:3, 3:0,4:0,5:0, 6:1,7:1,8:1, 9:2,10:2,11:2}
        df['season']      = df['month'].map(season_map)
        try:
            import holidays
            es_holidays  = holidays.Spain(years=range(2015, 2020))
            df['is_holiday'] = t.dt.date.astype(str).map(lambda d: 1 if d in [str(h) for h in es_holidays] else 0)
        except ImportError:
            df['is_holiday'] = 0
        return df

    def _add_cyclical(self, df):
        df['hour_sin']  = np.sin(2*np.pi*df['hour']/24)
        df['hour_cos']  = np.cos(2*np.pi*df['hour']/24)
        df['dow_sin']   = np.sin(2*np.pi*df['dow']/7)
        df['dow_cos']   = np.cos(2*np.pi*df['dow']/7)
        df['month_sin'] = np.sin(2*np.pi*df['month']/12)
        df['month_cos'] = np.cos(2*np.pi*df['month']/12)
        df['doy_sin']   = np.sin(2*np.pi*df['day_of_year']/365)
        df['doy_cos']   = np.cos(2*np.pi*df['day_of_year']/365)
        return df

    def _add_lags(self, df):
        for lag in self.lag_hours:
            df[f'lag_{lag}h'] = df[TARGET].shift(lag)
        return df

    def _add_rolling(self, df):
        shifted = df[TARGET].shift(1)
        df['rolling_mean_24h']  = shifted.rolling(24,  min_periods=12).mean()
        df['rolling_mean_168h'] = shifted.rolling(168, min_periods=84).mean()
        df['rolling_std_24h']   = shifted.rolling(24,  min_periods=12).std()
        df['rolling_max_24h']   = shifted.rolling(24,  min_periods=12).max()
        df['rolling_min_24h']   = shifted.rolling(24,  min_periods=12).min()
        return df

    def _add_weather(self, df):
        df['temp_squared']  = df['temp'] ** 2
        df['is_cold']       = (df['temp'] < 280).astype(int)
        df['is_hot']        = (df['temp'] > 298).astype(int)
        df['temp_humidity'] = df['temp'] * df['humidity'] / 100
        df['is_raining']    = (df['rain_1h'] > 0).astype(int)
        df['wind_chill']    = df['wind_speed'] * (1 - df['temp'] / 310)
        return df

print('EnergyFeatureTransformer ready ✅')

EnergyFeatureTransformer ready ✅


## 3. Prepare Features and Target

In [4]:
# Apply feature engineering to train set
fe = EnergyFeatureTransformer()
train_fe = fe.fit_transform(train_raw)

X_train = train_fe[ALL_FEATURES]
y_train = train_fe[TARGET]

print(f'X_train: {X_train.shape}')
print(f'y_train: {y_train.shape}')
print(f'y_train mean: {y_train.mean():,.0f} MW')

X_train: (24208, 40)
y_train: (24208,)
y_train mean: 28,563 MW


## 4. Helper: Compute Metrics

In [5]:
def compute_metrics(y_true, y_pred, label=''):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    if label:
        print(f'{label}:')
        print(f'  RMSE: {rmse:>10,.2f} MW')
        print(f'  MAE:  {mae:>10,.2f} MW')
        print(f'  R²:   {r2:>10.4f}')
        print(f'  MAPE: {mape:>10.2f} %')
    return {'rmse': rmse, 'mae': mae, 'r2': r2, 'mape': mape}

def cv_metrics(pipeline, X, y, cv, label=''):
    rmse_scores = -cross_val_score(pipeline, X, y, cv=cv, scoring='neg_root_mean_squared_error')
    mae_scores  = -cross_val_score(pipeline, X, y, cv=cv, scoring='neg_mean_absolute_error')
    r2_scores   =  cross_val_score(pipeline, X, y, cv=cv, scoring='r2')
    if label:
        print(f'{label} (5-fold TimeSeriesSplit CV):')
        print(f'  CV RMSE: {rmse_scores.mean():>10,.2f} ± {rmse_scores.std():,.2f} MW')
        print(f'  CV MAE:  {mae_scores.mean():>10,.2f} ± {mae_scores.std():,.2f} MW')
        print(f'  CV R²:   {r2_scores.mean():>10.4f} ± {r2_scores.std():.4f}')
    return {
        'cv_rmse_mean': rmse_scores.mean(), 'cv_rmse_std': rmse_scores.std(),
        'cv_mae_mean':  mae_scores.mean(),  'cv_mae_std':  mae_scores.std(),
        'cv_r2_mean':   r2_scores.mean(),   'cv_r2_std':   r2_scores.std(),
    }

print('Metric helpers ready ✅')

Metric helpers ready ✅


## 5. MLflow Setup

In [6]:
mlflow.set_experiment(MLFLOW_EXPERIMENT)
print(f'MLflow experiment: "{MLFLOW_EXPERIMENT}"')
print('Start MLflow UI with: mlflow ui')
print('Then open: http://127.0.0.1:5000')

2026/03/08 09:55:28 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/03/08 09:55:28 INFO mlflow.store.db.utils: Updating database tables
2026/03/08 09:55:30 INFO mlflow.tracking.fluent: Experiment with name 'energy-demand-prediction' does not exist. Creating a new experiment.


MLflow experiment: "energy-demand-prediction"
Start MLflow UI with: mlflow ui
Then open: http://127.0.0.1:5000


## 6. Naive Baseline — Predict Last Week Same Hour

In [7]:
with mlflow.start_run(run_name='naive_baseline_lag168h'):

    # Prediction = demand from exactly 168 hours ago (same hour last week)
    y_pred_naive = X_train['lag_168h']
    
    # Align index
    mask = y_pred_naive.notna()
    metrics = compute_metrics(y_train[mask], y_pred_naive[mask], label='Naive baseline')

    mlflow.log_param('model_type',   'Naive')
    mlflow.log_param('strategy',     'lag_168h (last week same hour)')
    mlflow.log_metric('train_rmse',  metrics['rmse'])
    mlflow.log_metric('train_mae',   metrics['mae'])
    mlflow.log_metric('train_r2',    metrics['r2'])
    mlflow.log_metric('train_mape',  metrics['mape'])

    print()
    print('✅ Naive baseline logged to MLflow')
    print('→ This is the floor. Any ML model must beat this.')

Naive baseline:
  RMSE:   3,766.94 MW
  MAE:    2,621.66 MW
  R²:       0.3090
  MAPE:       9.29 %

✅ Naive baseline logged to MLflow
→ This is the floor. Any ML model must beat this.


## 7. Ridge Baseline — No Lag Features (calendar + weather only)

In [8]:
CALENDAR_FEATURES = config['calendar_features']
CYCLICAL_FEATURES = config['cyclical_features']
WEATHER_ORIG      = config['weather_orig']
WEATHER_ENG       = config['weather_eng']
LOAD_FORECAST     = config['load_forecast']

SIMPLE_FEATURES = CALENDAR_FEATURES + CYCLICAL_FEATURES + WEATHER_ORIG + WEATHER_ENG + LOAD_FORECAST

with mlflow.start_run(run_name='ridge_no_lags'):

    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('model',  Ridge(alpha=1.0))
    ])

    metrics_cv = cv_metrics(pipeline, X_train[SIMPLE_FEATURES], y_train, TSCV,
                             label='Ridge (no lags)')

    pipeline.fit(X_train[SIMPLE_FEATURES], y_train)
    train_metrics = compute_metrics(y_train, pipeline.predict(X_train[SIMPLE_FEATURES]))

    mlflow.log_param('model_type',    'Ridge')
    mlflow.log_param('features',      'calendar + weather (no lags)')
    mlflow.log_param('alpha',         1.0)
    mlflow.log_param('cv_strategy',   'TimeSeriesSplit n=5 gap=24')
    mlflow.log_metric('train_rmse',   train_metrics['rmse'])
    mlflow.log_metric('train_r2',     train_metrics['r2'])
    mlflow.log_metric('cv_rmse_mean', metrics_cv['cv_rmse_mean'])
    mlflow.log_metric('cv_rmse_std',  metrics_cv['cv_rmse_std'])
    mlflow.log_metric('cv_r2_mean',   metrics_cv['cv_r2_mean'])
    mlflow.log_metric('overfit_gap',  train_metrics['r2'] - metrics_cv['cv_r2_mean'])
    mlflow.sklearn.log_model(pipeline, artifact_path='model')

    print()
    print('✅ Ridge (no lags) logged to MLflow')

2026/03/08 09:55:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Ridge (no lags) (5-fold TimeSeriesSplit CV):
  CV RMSE:     901.80 ± 922.42 MW
  CV MAE:      750.50 ± 855.62 MW
  CV R²:       0.9207 ± 0.1389


2026/03/08 09:55:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



✅ Ridge (no lags) logged to MLflow


## 8. Ridge Baseline — Full Features (with lags)

In [9]:
with mlflow.start_run(run_name='ridge_full_features'):

    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('model',  Ridge(alpha=1.0))
    ])

    metrics_cv = cv_metrics(pipeline, X_train, y_train, TSCV,
                             label='Ridge (full features + lags)')

    pipeline.fit(X_train, y_train)
    train_metrics = compute_metrics(y_train, pipeline.predict(X_train))

    mlflow.log_param('model_type',    'Ridge')
    mlflow.log_param('features',      'full — calendar + weather + lags + rolling')
    mlflow.log_param('n_features',    len(ALL_FEATURES))
    mlflow.log_param('alpha',         1.0)
    mlflow.log_param('cv_strategy',   'TimeSeriesSplit n=5 gap=24')
    mlflow.log_metric('train_rmse',   train_metrics['rmse'])
    mlflow.log_metric('train_r2',     train_metrics['r2'])
    mlflow.log_metric('cv_rmse_mean', metrics_cv['cv_rmse_mean'])
    mlflow.log_metric('cv_rmse_std',  metrics_cv['cv_rmse_std'])
    mlflow.log_metric('cv_r2_mean',   metrics_cv['cv_r2_mean'])
    mlflow.log_metric('overfit_gap',  train_metrics['r2'] - metrics_cv['cv_r2_mean'])
    mlflow.sklearn.log_model(pipeline, artifact_path='model')

    print()
    print('✅ Ridge (full features) logged to MLflow')
    print()
    print('→ This is the REAL baseline. Notebook 05 models must beat this CV RMSE.')

Ridge (full features + lags) (5-fold TimeSeriesSplit CV):
  CV RMSE:     910.97 ± 984.81 MW
  CV MAE:      763.08 ± 903.63 MW
  CV R²:       0.9144 ± 0.1536


2026/03/08 09:55:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/08 09:55:40 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



✅ Ridge (full features) logged to MLflow

→ This is the REAL baseline. Notebook 05 models must beat this CV RMSE.


## 9. Baseline Summary

In [10]:
print('=' * 55)
print('BASELINE SUMMARY')
print('=' * 55)
print('Model                         | CV RMSE    | CV R²')
print('------------------------------|------------|------')
print('Naive (lag_168h)              | check MLflow (train only)')
print('Ridge (no lags)               | check MLflow')
print('Ridge (full features + lags)  | check MLflow  ← REAL BASELINE')
print()
print('Open MLflow UI to compare: mlflow ui')
print()
print('Note: RMSE is in MW. Target mean ~28,700 MW.')
print('A RMSE of 1000 MW = ~3.5% error — reasonable for hourly forecasting.')

BASELINE SUMMARY
Model                         | CV RMSE    | CV R²
------------------------------|------------|------
Naive (lag_168h)              | check MLflow (train only)
Ridge (no lags)               | check MLflow
Ridge (full features + lags)  | check MLflow  ← REAL BASELINE

Open MLflow UI to compare: mlflow ui

Note: RMSE is in MW. Target mean ~28,700 MW.
A RMSE of 1000 MW = ~3.5% error — reasonable for hourly forecasting.
